[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nursnaaz/zero-to-genai-engineer/blob/main/10_RAG/notebooks/13_capstone_production_rag_chatbot.ipynb)

# Capstone — Assembling a Production RAG Chatbot

Notebooks 01–08 each gave you **one piece** of the retrieval stack:

| Notebook | Piece |
|---|---|
| 01 | Why retrieval beats a bare LLM |
| 02 / 03 | Parsing + chunking (LangChain / LlamaIndex) |
| 04 | Embeddings — the geometry of meaning |
| 05 | Vector databases — FAISS → Chroma → Pinecone |
| 06 | Sparse retrieval — BM25 |
| 07 | Hybrid search — dense + sparse fused with RRF |
| 08 | Reranking — a cross-encoder precision pass |

**Today those pieces stop being separate demos.** We assemble every one of them, in order,
into a single class — `ProductionRAGChatbot` — that:

1. Ingests **your own uploaded document** (PDF, DOCX, TXT, or MD)
2. Chunks it and builds **both** a dense (Chroma) and sparse (BM25) index
3. Retrieves with **RRF fusion**, then **reranks** with a cross-encoder
4. **Refuses to answer** when nothing relevant was retrieved (a real guardrail, not a demo one)
5. Remembers the conversation, so follow-up questions ("what about *its* refund policy?") resolve correctly
6. Answers **only** from retrieved text, with numbered citations back to source + page

Then, in the last section, we export the exact same class into a small **Streamlit app**
so it stops being a notebook and becomes something you can put in front of another person.

> **How to work through this notebook:** run every cell top to bottom once with the sample
> report in `data/sample_report.pdf`, then go back to Section 3, upload *your own* file, and
> re-run from there. The whole point of today is watching this pipeline work on a document
> you picked, not ours.


## 0. Install dependencies

Run this first, then (on a fresh environment) restart the kernel and re-run top to bottom.

In [ ]:
# Install dependencies into the ACTIVE kernel (idempotent — skips what's already there).
%pip install -q \
    langchain langchain-core langchain-openai langchain-community langchain-text-splitters \
    langchain-pymupdf4llm pypdf docx2txt \
    sentence-transformers bm25s PyStemmer chromadb \
    python-dotenv numpy<2
print("\u2705 Dependencies ready. If this was a fresh install, restart the kernel now, then re-run.")

## 1. Setup

Same pattern as every notebook in this series: load `OPENAI_API_KEY` from the `.env` file
one directory up, and quiet down library warnings so the teaching output stays readable.


In [ ]:
import warnings, os, sys
warnings.filterwarnings("ignore")
import logging
for _n in ("httpx", "openai", "httpcore", "sentence_transformers", "transformers", "chromadb"):
    logging.getLogger(_n).setLevel(logging.ERROR)

from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")
DATA = Path.cwd() / "data"

print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("sample files available:", sorted(p.name for p in DATA.iterdir() if p.is_file() and not p.name.startswith("_")))

## 2. The cold open — watch a naive RAG fail

Before building anything, let's see the actual problem. Below is the simplest RAG anyone
ever ships: embed every chunk, embed the query, grab the **single closest chunk**, stuff it
into a prompt, done. No hybrid search, no reranking, no citations, no guardrail — this is
what most "RAG in 10 lines" tutorials show you, on the fixed sample report in `data/`.


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

try:
    from langchain_pymupdf4llm import PyMuPDF4LLMLoader
    _HAS_PYMUPDF4LLM = True
except ImportError:
    _HAS_PYMUPDF4LLM = False

# Same parsing AND chunking the real pipeline uses later — the ONLY thing "naive" here is the
# retrieval itself: top-1, dense-only, no rerank, no guardrail. That isolates the one variable
# this class is actually about, instead of accidentally comparing two different chunkers too.
_cold_encoder = SentenceTransformer("BAAI/bge-small-en-v1.5")
_cold_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

_cold_path = DATA / "sample_report.pdf"
_cold_page = (PyMuPDF4LLMLoader(str(_cold_path), table_strategy="lines").load()[0].page_content
              if _HAS_PYMUPDF4LLM else PyPDFLoader(str(_cold_path)).load()[0].page_content)
_cold_chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80).split_text(_cold_page)
_cold_emb = _cold_encoder.encode(_cold_chunks, normalize_embeddings=True, show_progress_bar=False)

def naive_rag(query):
    q = _cold_encoder.encode([query], normalize_embeddings=True)[0]
    top = int(np.argmax(q @ _cold_emb.T))          # ONE chunk, nothing else
    prompt = f"Answer using this context:\n\n{_cold_chunks[top]}\n\nQuestion: {query}\nAnswer:"
    return _cold_llm.invoke(prompt).content.strip()

COLD_OPEN_QUERY = "What were total shipments, and how does the Northeast warehouse migration relate to it?"
print("Q:", COLD_OPEN_QUERY)
print("Naive RAG answer:", naive_rag(COLD_OPEN_QUERY))

**Watch it miss.** The naive answer talks about "8.4% growth" and never states the actual
total — 601,710 — because top-1 retrieval grabbed the *intro paragraph* (which mentions the
growth rate) instead of the *table* a few lines later (which has the real number). The model
isn't lying; it just confidently answers with the one chunk it was handed, and that chunk
happened not to contain what was asked. That's the dangerous failure mode — not silence,
a **plausible-sounding non-answer**.

**That's the whole class today.** Every section from here on fixes one piece of this: chunking
that preserves structure, retrieving more than one chunk, fusing multiple retrieval signals,
reranking for precision, and generating with citations you can actually check. At the end,
we'll ask this exact question again through the finished pipeline and compare.


## 3. Upload your document — the pipeline works on *your* file, not ours

In Colab this opens a real file picker. Running locally in Jupyter, it falls back to the
sample report so the notebook still runs end to end with no extra steps — but **come back
here and swap in your own PDF/DOCX/TXT once you've seen it work once.**

Supported formats: `.pdf`, `.docx`, `.txt`, `.md`. Multiple files are fine — everything gets
pooled into one index, same as a real knowledge base.


In [ ]:
def get_uploaded_files():
    """Colab: real upload widget. Local Jupyter: fall back to the sample report."""
    try:
        from google.colab import files
        print("Choose one or more files (.pdf / .docx / .txt / .md) ...")
        uploaded = files.upload()
        return [str(Path(name).resolve()) for name in uploaded.keys()]
    except ImportError:
        default = DATA / "sample_report.pdf"
        print(f"Not running in Colab — using the sample file: {default.name}")
        print("(Re-run this cell in Colab, or edit the path below, to use your own document.)")
        return [str(default)]

FILE_PATHS = get_uploaded_files()
print("\nFiles to ingest:", FILE_PATHS)

### Multi-format parsing (generalizing Notebook 02)

Notebook 02 showed PDF, HTML, Word, and OCR loaders one at a time. A production ingester
needs to **pick the right loader from the file extension automatically** — a user uploading
a document doesn't know or care which LangChain loader class handles it.

`PyMuPDF4LLMLoader` is the layout-aware PDF loader from Notebook 02 (tables survive as
Markdown); we fall back to the plain `PyPDFLoader` if it isn't installed.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader

try:
    from langchain_pymupdf4llm import PyMuPDF4LLMLoader
    HAS_PYMUPDF4LLM = True
except ImportError:
    HAS_PYMUPDF4LLM = False


def load_document(path: str) -> list[dict]:
    """Parse one file into a list of {text, source, page} records — one per page."""
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".pdf":
        docs = (PyMuPDF4LLMLoader(str(path), table_strategy="lines").load()
                if HAS_PYMUPDF4LLM else PyPDFLoader(str(path)).load())
    elif suffix == ".docx":
        docs = Docx2txtLoader(str(path)).load()
    elif suffix in (".txt", ".md"):
        text = path.read_text(encoding="utf-8", errors="ignore")
        return [{"text": text, "source": path.name, "page": 1}]
    else:
        raise ValueError(f"Unsupported file type: {suffix!r} — use .pdf, .docx, .txt, or .md")

    return [
        {"text": d.page_content, "source": path.name, "page": d.metadata.get("page", 0) + 1}
        for d in docs if d.page_content.strip()
    ]


pages = []
for fp in FILE_PATHS:
    pages.extend(load_document(fp))

print(f"Parsed {len(pages)} page(s) from {len(FILE_PATHS)} file(s)")
print("Preview of page 1:\n", pages[0]["text"][:400])

## 4. Chunking — the sane default, with citations preserved

Notebook 02 compared six chunking strategies. For a general-purpose document chatbot,
**recursive character splitting** is the production default — it tries paragraph, then
line, then word boundaries, so chunks rarely cut a sentence in half. We carry `source`
and `page` through every chunk, because without that, citations in the final answer are
impossible.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_documents(pages, chunk_size=500, chunk_overlap=80):
    """Split parsed pages into overlapping chunks; keep source + page for citations."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = []
    for page in pages:
        for piece in splitter.split_text(page["text"]):
            chunks.append({"text": piece, "source": page["source"], "page": page["page"]})
    return chunks

CHUNK_SIZE, CHUNK_OVERLAP = 500, 80   # characters — tune these for your document type
chunks = chunk_documents(pages, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"{len(pages)} page(s) \u2192 {len(chunks)} chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")
print("chunk 0:", chunks[0])

## 5. Building the hybrid index — dense (Notebooks 04/05) + sparse (Notebook 06) together

This is Notebooks 04, 05, and 06 in one class:

- **Dense**: `bge-small-en-v1.5` (Notebook 07's small, fast bi-encoder) embeds every chunk,
  stored in a **Chroma** collection (Notebook 05) — production-shaped, with metadata attached.
- **Sparse**: the same chunk text is indexed with **BM25** (Notebook 06) via `bm25s`.

Both indexes are built over the **same chunk list, in the same order** — that's what lets us
fuse their rankings later.


In [ ]:
import uuid
import numpy as np
import bm25s
from Stemmer import Stemmer
from sentence_transformers import SentenceTransformer
import chromadb


class HybridIndex:
    """Dense (Chroma + bge-small) + sparse (BM25) index, fused with RRF at query time."""

    def __init__(self, collection_name="rag_chatbot"):
        self.encoder = SentenceTransformer("BAAI/bge-small-en-v1.5")
        self.stemmer = Stemmer("english")
        # In-memory Chroma client for the notebook demo — swap for
        # chromadb.PersistentClient(path=...) to survive a restart, as in Notebook 05.
        self.client = chromadb.Client()
        self.collection = self.client.get_or_create_collection(
            collection_name, metadata={"hnsw:space": "cosine"}
        )
        self.chunks = []   # aligned 1:1 with the BM25 corpus order
        self.bm25 = None
        self._doc_emb = None

    def _tokenize(self, texts):
        return bm25s.tokenize(texts, stopwords="en", stemmer=self.stemmer.stemWords, return_ids=False, show_progress=False)

    def build(self, chunks):
        self.chunks = chunks
        texts = [c["text"] for c in chunks]
        ids = [str(uuid.uuid4()) for _ in chunks]
        for c, cid in zip(chunks, ids):
            c["id"] = cid

        # Dense: bge-small embeddings \u2192 Chroma (same model for index AND query — Notebook 04's iron rule)
        vecs = self.encoder.encode(texts, normalize_embeddings=True, show_progress_bar=False)
        metadatas = [{"source": c["source"], "page": c["page"]} for c in chunks]
        self.collection.add(embeddings=vecs.tolist(), documents=texts, metadatas=metadatas, ids=ids)
        self._doc_emb = vecs

        # Sparse: BM25 over the identical chunk order
        self.bm25 = bm25s.BM25(method="lucene")
        self.bm25.index(self._tokenize(texts), show_progress=False)

    def _dense_order(self, query):
        q = self.encoder.encode([query], normalize_embeddings=True)[0]
        return list(np.argsort(q @ self._doc_emb.T)[::-1])

    def _sparse_order(self, query):
        scores = self.bm25.get_scores(self._tokenize([query])[0])
        return list(np.argsort(scores)[::-1])

    def search(self, query, k=8, rrf_k=60):
        """Reciprocal Rank Fusion of the dense and sparse rankings (Notebook 07)."""
        rankings = [self._dense_order(query), self._sparse_order(query)]
        scores = {}
        for ranking in rankings:
            for rank, idx in enumerate(ranking):
                scores[idx] = scores.get(idx, 0.0) + 1.0 / (rrf_k + rank)
        fused = sorted(scores, key=scores.get, reverse=True)[:k]
        return [self.chunks[i] for i in fused]


index = HybridIndex()
index.build(chunks)
print(f"Indexed {len(chunks)} chunks \u2014 dense (Chroma/bge-small) + sparse (BM25) both ready")

## 6. Retrieval — proving fusion beats either retriever alone

Same proof as Notebook 07, now running on **your own document**. Ask something where the
exact words might not be in the text (dense should catch it) and something with an exact
term or number (sparse should catch it) — RRF should hold up on both.


In [ ]:
DEMO_QUERY = "What are the main findings or key numbers in this document?"

dense_top = index.chunks[index._dense_order(DEMO_QUERY)[0]]
sparse_top = index.chunks[index._sparse_order(DEMO_QUERY)[0]]
fused = index.search(DEMO_QUERY, k=8)

print("query:", DEMO_QUERY)
print("\ndense-only top chunk: ", dense_top["text"][:150].replace("\n", " "))
print("sparse-only top chunk:", sparse_top["text"][:150].replace("\n", " "))
print(f"\nfused (RRF) returned {len(fused)} candidates for reranking; top one:")
print(" ", fused[0]["text"][:150].replace("\n", " "))

## 7. Reranking — the precision second stage (Notebook 08)

RRF fusion is good at **recall** (getting the right chunk *into* the top-k), not always at
**precision** (putting it at #1). The cross-encoder reruns the top-k candidates with a model
that reads the query and each chunk **together**, then keeps only the best few — the same
`top_n` chunks that go into the final prompt.


In [ ]:
from sentence_transformers import CrossEncoder

class Reranker:
    """Cross-encoder second stage (Notebook 08) over the fused candidates."""

    def __init__(self, model_name="cross-encoder/ms-marco-MiniLM-L6-v2"):
        try:
            self.model = CrossEncoder(model_name)
        except Exception:
            self.model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

    def rerank(self, query, candidates, top_n=4):
        pairs = [(query, c["text"]) for c in candidates]
        scores = self.model.predict(pairs)
        ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
        return [(c, float(s)) for c, s in ranked[:top_n]]


reranker = Reranker()
reranked = reranker.rerank(DEMO_QUERY, fused, top_n=4)

print("Reranked top 4 (score, source, preview):")
for c, score in reranked:
    print(f"  {score:6.2f}  {c['source']} p.{c['page']}  {c['text'][:90].strip()!r}")

## 8. Grounded generation with citations

The model only ever sees the reranked chunks — **numbered**, with source and page — and is
instructed to cite every claim `[1]`, `[2]`, etc. This is what turns "an LLM that read some
text" into an answer you can actually verify.


In [ ]:
from langchain_openai import ChatOpenAI

ANSWER_PROMPT = """You are a careful assistant answering questions ONLY using the numbered sources below.
- Cite sources inline like [1], [2] after every claim they support.
- If the sources don't contain the answer, say so plainly \u2014 never guess.

Sources:
{sources}

Conversation so far:
{history}

Question: {question}

Answer (with inline citations):"""

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_sources(reranked):
    return "\n".join(f"[{i+1}] ({c['source']}, p.{c['page']}) {c['text']}" for i, (c, _) in enumerate(reranked))

prompt = ANSWER_PROMPT.format(sources=format_sources(reranked), history="(none yet)", question=DEMO_QUERY)
answer = llm.invoke(prompt).content.strip()

print("Q:", DEMO_QUERY)
print("\nA:", answer)

## 9. The groundedness guardrail — refuse rather than hallucinate

Ask something the document plainly doesn't cover. Without a guardrail, the LLM will often
still produce a fluent, confident, **wrong** answer — the exact failure mode Notebook 01
opened with. The fix is cheap: look at the reranker's own top score. If even the *best*
candidate scores low, there's nothing to answer from, so refuse instead of guessing.

> The threshold below (`MIN_RERANK_SCORE`) is a heuristic, not a law of nature — cross-encoder
> scores aren't calibrated probabilities, and they run noticeably lower for broad "summarize
> this" questions than for specific factual ones, even when both are perfectly answerable.
> For `ms-marco-MiniLM-L6-v2`, clearly-irrelevant queries tend to floor out around **-11**,
> while on-topic queries — vague or specific — usually land above **-10**. That gap is why the
> default below is `-9.5`, not something tighter like `-3`. Calibrate on your own corpus: run
> a few known-answerable and known-unanswerable queries, look at the score gap, and set the
> threshold in between.


In [ ]:
MIN_RERANK_SCORE = -9.5   # tune per corpus \u2014 see note above

def answer_or_refuse(query, reranked):
    if not reranked or reranked[0][1] < MIN_RERANK_SCORE:
        return "I don't have enough information in the uploaded document(s) to answer that confidently.", []
    prompt = ANSWER_PROMPT.format(sources=format_sources(reranked), history="(none yet)", question=query)
    return llm.invoke(prompt).content.strip(), reranked

OUT_OF_SCOPE_QUERY = "What is the capital of France?"
candidates = index.search(OUT_OF_SCOPE_QUERY, k=8)
reranked_oos = reranker.rerank(OUT_OF_SCOPE_QUERY, candidates, top_n=4)

print("top rerank score for an out-of-scope query:", round(reranked_oos[0][1], 2))
answer, used = answer_or_refuse(OUT_OF_SCOPE_QUERY, reranked_oos)
print("\nQ:", OUT_OF_SCOPE_QUERY)
print("A:", answer)

## 10. Conversational memory — the condense-question pattern

A real chat has follow-ups: *"What about its refund policy?"* means nothing to a retriever
on its own — "its" needs to resolve to whatever was being discussed. The production fix is
a **condense-question step**: before retrieving, ask the LLM to rewrite the follow-up as a
standalone question using the last few turns of history. Retrieval then runs on the rewritten
question, never on the raw follow-up.


In [ ]:
CONDENSE_PROMPT = """Given the recent conversation and a follow-up question, decide whether the
follow-up depends on the conversation (uses a pronoun or implicit reference like "it", "that",
"those", "the previous one") to make sense.

- If it DOES depend on the conversation, rewrite it as a standalone question that replaces the
  reference with the actual thing it refers to.
- If it does NOT depend on the conversation — including if it's simply a new question on a
  different topic — return it EXACTLY UNCHANGED. Do not "helpfully" relate an unrelated
  question back to the previous topic.

Conversation:
{history}

Follow-up question: {question}

Standalone question:"""

def condense_question(question, history_turns):
    if not history_turns:
        return question
    history_text = "\n".join(f"{role}: {content}" for role, content in history_turns[-6:])
    return llm.invoke(CONDENSE_PROMPT.format(history=history_text, question=question)).content.strip()

# Simulate a two-turn conversation
history_turns = [("user", "What does this document say about the budget?"),
                  ("assistant", "It covers quarterly budget allocations across regions. [1]")]
follow_up = "What about its risks?"

standalone = condense_question(follow_up, history_turns)
print("raw follow-up:      ", follow_up)
print("standalone question:", standalone)

## 11. Assembling everything — `ProductionRAGChatbot`

One class, six ideas: ingest \u2192 hybrid index \u2192 fuse \u2192 rerank \u2192 guardrail \u2192 grounded,
cited, memory-aware generation. Every method below is code you already ran above — this cell
just wires it into something you can call `.chat()` on, turn after turn.


In [ ]:
from dataclasses import dataclass


@dataclass
class ChatTurn:
    role: str
    content: str


class ProductionRAGChatbot:
    """
    Assembles Notebooks 01-08 into one pipeline:
    parse -> chunk -> hybrid index (dense+sparse) -> RRF fuse -> cross-encoder rerank
    -> grounded, cited generation -> guarded against ungrounded answers -> memory-aware.
    """

    def __init__(self, chunk_size=500, chunk_overlap=80, top_k=8, top_n=4,
                 min_rerank_score=-9.5, model="gpt-4o-mini"):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.top_k = top_k
        self.top_n = top_n
        self.min_rerank_score = min_rerank_score
        self.index = HybridIndex()
        self.reranker = Reranker()
        self.llm = ChatOpenAI(model=model, temperature=0)
        self.history: list[ChatTurn] = []

    def ingest(self, file_paths):
        """Parse + chunk one or more files, then build the hybrid index."""
        all_pages = []
        for path in file_paths:
            all_pages.extend(load_document(path))
        chunks = chunk_documents(all_pages, self.chunk_size, self.chunk_overlap)
        self.index.build(chunks)
        return {"files": len(file_paths), "pages": len(all_pages), "chunks": len(chunks)}

    def _history_pairs(self):
        return [(t.role, t.content) for t in self.history]

    def _condense(self, question):
        return condense_question(question, self._history_pairs())

    def chat(self, question):
        standalone = self._condense(question)
        candidates = self.index.search(standalone, k=self.top_k)
        reranked = self.reranker.rerank(standalone, candidates, top_n=self.top_n)

        if not reranked or reranked[0][1] < self.min_rerank_score:
            answer = "I don't have enough information in the uploaded document(s) to answer that confidently."
            sources = []
        else:
            history_text = "\n".join(f"{t.role}: {t.content}" for t in self.history[-6:])
            prompt = ANSWER_PROMPT.format(
                sources=format_sources(reranked), history=history_text, question=standalone
            )
            answer = self.llm.invoke(prompt).content.strip()
            sources = [
                {"rank": i + 1, "source": c["source"], "page": c["page"], "score": round(s, 3), "text": c["text"]}
                for i, (c, s) in enumerate(reranked)
            ]

        self.history.append(ChatTurn("user", question))
        self.history.append(ChatTurn("assistant", answer))
        return {"answer": answer, "sources": sources, "standalone_question": standalone}

    def naive_chat(self, question):
        """The Section 2 baseline, reusing the SAME index: top-1 dense-only, no rerank,
        no guardrail, no citations, no memory. For side-by-side comparison only."""
        order = self.index._dense_order(question)
        top_chunk = self.index.chunks[order[0]]
        prompt = f"Answer using this context:\n\n{top_chunk['text']}\n\nQuestion: {question}\nAnswer:"
        answer = self.llm.invoke(prompt).content.strip()
        return {"answer": answer, "source": {"source": top_chunk["source"], "page": top_chunk["page"]}}


print("ProductionRAGChatbot ready")

## 12. Take it for a spin

Ingest your uploaded file(s) into a fresh bot, then run a small conversation: a normal
question, a follow-up that only makes sense with memory, an out-of-scope question that
should trigger the guardrail — **and the cold-open question from Section 2**, so you can
compare the naive answer against the finished pipeline directly.


In [ ]:
bot = ProductionRAGChatbot()
stats = bot.ingest(FILE_PATHS)
print("Ingested:", stats, "\n")

conversation = [
    "Summarize what this document is about in two sentences.",
    "What specific numbers or figures does it mention?",
    "What about its risks or limitations?",     # follow-up — needs memory to resolve "its"
    "What's the population of Mars?",           # out-of-scope — should trigger the guardrail
    COLD_OPEN_QUERY,                             # the exact question the naive baseline missed
]

for turn in conversation:
    result = bot.chat(turn)
    print(f"You: {turn}")
    if result["standalone_question"] != turn:
        print(f"  (resolved to: {result['standalone_question']!r})")
    print(f"Bot: {result['answer']}")
    if result["sources"]:
        cited = ", ".join(f"[{s['rank']}] {s['source']} p.{s['page']}" for s in result["sources"])
        print(f"  Sources: {cited}")
    print()

### Cold open, revisited

Same question, side by side:

| | Naive RAG (Section 2) | `ProductionRAGChatbot` (just now) |
|---|---|---|
| Retrieval | Top-1 chunk, dense-only | Hybrid (dense+BM25) → RRF → reranked top-n |
| Total shipments figure | Never stated — wrong chunk retrieved, answer conflates growth % with the total | 601,710, cited |
| Citations | None | `[source, page]` on every claim |
| Verifiable? | No — take it on faith | Yes — check the cited chunk yourself |

That gap — one number missing from an honest-sounding answer — is exactly the kind of failure
that's invisible until someone downstream relies on it. Everything built in this notebook
exists to close that gap.


## 13. Ship it — export the pipeline, then wrap it in a chat UI

A notebook is a workbench. To hand this to someone else, it needs a face. The cell below
writes the **exact class you just built** out to `production_rag_chatbot/rag_pipeline.py`.
A companion `production_rag_chatbot/app.py` (already in this repo, next to this notebook)
imports that module and wraps it in a Streamlit chat interface — file uploader, chat bubbles,
an expandable "Sources" panel per answer, and sidebar controls for chunk size, top-k, and the
guardrail threshold.

Run it after this cell:

```bash
cd production_rag_chatbot
streamlit run app.py
```


In [ ]:
RAG_PIPELINE_SOURCE = '"""\nProduction RAG pipeline \\u2014 the exact class assembled in Notebook 13, saved as an\nimportable module so both the notebook and the Streamlit app (app.py) share one\nimplementation instead of drifting apart.\n\nPipeline: parse -> chunk -> hybrid index (dense Chroma/bge-small + sparse BM25)\n-> RRF fuse -> cross-encoder rerank -> grounded, cited generation\n-> groundedness guardrail -> conversational memory (condense-question pattern).\n"""\n\nimport os\nimport uuid\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport bm25s\nfrom Stemmer import Stemmer\nfrom sentence_transformers import SentenceTransformer, CrossEncoder\nimport chromadb\n\nfrom langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader\nfrom langchain_text_splitters import RecursiveCharacterTextSplitter\nfrom langchain_openai import ChatOpenAI\n\ntry:\n    from langchain_pymupdf4llm import PyMuPDF4LLMLoader\n    HAS_PYMUPDF4LLM = True\nexcept ImportError:\n    HAS_PYMUPDF4LLM = False\n\n\nANSWER_PROMPT = \'\'\'You are a careful assistant answering questions ONLY using the numbered sources below.\n- Cite sources inline like [1], [2] after every claim they support.\n- If the sources don\\\'t contain the answer, say so plainly \\u2014 never guess.\n\nSources:\n{sources}\n\nConversation so far:\n{history}\n\nQuestion: {question}\n\nAnswer (with inline citations):\'\'\'\n\nCONDENSE_PROMPT = \'\'\'Given the recent conversation and a follow-up question, decide whether the\nfollow-up depends on the conversation (uses a pronoun or implicit reference like "it", "that",\n"those", "the previous one") to make sense.\n\n- If it DOES depend on the conversation, rewrite it as a standalone question that replaces the\n  reference with the actual thing it refers to.\n- If it does NOT depend on the conversation \\u2014 including if it\\\'s simply a new question on a\n  different topic \\u2014 return it EXACTLY UNCHANGED. Do not "helpfully" relate an unrelated\n  question back to the previous topic.\n\nConversation:\n{history}\n\nFollow-up question: {question}\n\nStandalone question:\'\'\'\n\n\ndef load_document(path: str) -> list[dict]:\n    """Parse one file into a list of {text, source, page} records \\u2014 one per page."""\n    path = Path(path)\n    suffix = path.suffix.lower()\n\n    if suffix == ".pdf":\n        docs = (PyMuPDF4LLMLoader(str(path), table_strategy="lines").load()\n                if HAS_PYMUPDF4LLM else PyPDFLoader(str(path)).load())\n    elif suffix == ".docx":\n        docs = Docx2txtLoader(str(path)).load()\n    elif suffix in (".txt", ".md"):\n        text = path.read_text(encoding="utf-8", errors="ignore")\n        return [{"text": text, "source": path.name, "page": 1}]\n    else:\n        raise ValueError(f"Unsupported file type: {suffix!r} \\u2014 use .pdf, .docx, .txt, or .md")\n\n    return [\n        {"text": d.page_content, "source": path.name, "page": d.metadata.get("page", 0) + 1}\n        for d in docs if d.page_content.strip()\n    ]\n\n\ndef chunk_documents(pages, chunk_size=500, chunk_overlap=80):\n    """Split parsed pages into overlapping chunks; keep source + page for citations."""\n    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)\n    chunks = []\n    for page in pages:\n        for piece in splitter.split_text(page["text"]):\n            chunks.append({"text": piece, "source": page["source"], "page": page["page"]})\n    return chunks\n\n\nclass HybridIndex:\n    """Dense (Chroma + bge-small) + sparse (BM25) index, fused with RRF at query time."""\n\n    def __init__(self, collection_name="rag_chatbot"):\n        self.encoder = SentenceTransformer("BAAI/bge-small-en-v1.5")\n        self.stemmer = Stemmer("english")\n        self.client = chromadb.Client()\n        self.collection = self.client.get_or_create_collection(\n            collection_name, metadata={"hnsw:space": "cosine"}\n        )\n        self.chunks = []\n        self.bm25 = None\n        self._doc_emb = None\n\n    def _tokenize(self, texts):\n        return bm25s.tokenize(texts, stopwords="en", stemmer=self.stemmer.stemWords, return_ids=False, show_progress=False)\n\n    def build(self, chunks):\n        self.chunks = chunks\n        texts = [c["text"] for c in chunks]\n        ids = [str(uuid.uuid4()) for _ in chunks]\n        for c, cid in zip(chunks, ids):\n            c["id"] = cid\n\n        vecs = self.encoder.encode(texts, normalize_embeddings=True, show_progress_bar=False)\n        metadatas = [{"source": c["source"], "page": c["page"]} for c in chunks]\n        self.collection.add(embeddings=vecs.tolist(), documents=texts, metadatas=metadatas, ids=ids)\n        self._doc_emb = vecs\n\n        self.bm25 = bm25s.BM25(method="lucene")\n        self.bm25.index(self._tokenize(texts), show_progress=False)\n\n    def _dense_order(self, query):\n        q = self.encoder.encode([query], normalize_embeddings=True)[0]\n        return list(np.argsort(q @ self._doc_emb.T)[::-1])\n\n    def _sparse_order(self, query):\n        scores = self.bm25.get_scores(self._tokenize([query])[0])\n        return list(np.argsort(scores)[::-1])\n\n    def search(self, query, k=8, rrf_k=60):\n        """Reciprocal Rank Fusion of the dense and sparse rankings."""\n        rankings = [self._dense_order(query), self._sparse_order(query)]\n        scores = {}\n        for ranking in rankings:\n            for rank, idx in enumerate(ranking):\n                scores[idx] = scores.get(idx, 0.0) + 1.0 / (rrf_k + rank)\n        fused = sorted(scores, key=scores.get, reverse=True)[:k]\n        return [self.chunks[i] for i in fused]\n\n\nclass Reranker:\n    """Cross-encoder second stage over the fused candidates."""\n\n    def __init__(self, model_name="cross-encoder/ms-marco-MiniLM-L6-v2"):\n        try:\n            self.model = CrossEncoder(model_name)\n        except Exception:\n            self.model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")\n\n    def rerank(self, query, candidates, top_n=4):\n        pairs = [(query, c["text"]) for c in candidates]\n        scores = self.model.predict(pairs)\n        ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)\n        return [(c, float(s)) for c, s in ranked[:top_n]]\n\n\ndef format_sources(reranked):\n    return "\\n".join(f"[{i + 1}] ({c[\'source\']}, p.{c[\'page\']}) {c[\'text\']}" for i, (c, _) in enumerate(reranked))\n\n\n@dataclass\nclass ChatTurn:\n    role: str\n    content: str\n\n\nclass ProductionRAGChatbot:\n    """\n    parse -> chunk -> hybrid index (dense+sparse) -> RRF fuse -> cross-encoder rerank\n    -> grounded, cited generation -> guarded against ungrounded answers -> memory-aware.\n    """\n\n    def __init__(self, chunk_size=500, chunk_overlap=80, top_k=8, top_n=4,\n                 min_rerank_score=-9.5, model="gpt-4o-mini"):\n        self.chunk_size = chunk_size\n        self.chunk_overlap = chunk_overlap\n        self.top_k = top_k\n        self.top_n = top_n\n        self.min_rerank_score = min_rerank_score\n        self.index = HybridIndex()\n        self.reranker = Reranker()\n        self.llm = ChatOpenAI(model=model, temperature=0)\n        self.history: list[ChatTurn] = []\n\n    def ingest(self, file_paths):\n        """Parse + chunk one or more files, then build the hybrid index."""\n        all_pages = []\n        for path in file_paths:\n            all_pages.extend(load_document(path))\n        chunks = chunk_documents(all_pages, self.chunk_size, self.chunk_overlap)\n        self.index.build(chunks)\n        return {"files": len(file_paths), "pages": len(all_pages), "chunks": len(chunks)}\n\n    def _history_pairs(self):\n        return [(t.role, t.content) for t in self.history]\n\n    def _condense(self, question):\n        if not self.history:\n            return question\n        history_text = "\\n".join(f"{role}: {content}" for role, content in self._history_pairs()[-6:])\n        return self.llm.invoke(CONDENSE_PROMPT.format(history=history_text, question=question)).content.strip()\n\n    def chat(self, question):\n        standalone = self._condense(question)\n        candidates = self.index.search(standalone, k=self.top_k)\n        reranked = self.reranker.rerank(standalone, candidates, top_n=self.top_n)\n\n        if not reranked or reranked[0][1] < self.min_rerank_score:\n            answer = "I don\\\'t have enough information in the uploaded document(s) to answer that confidently."\n            sources = []\n        else:\n            history_text = "\\n".join(f"{t.role}: {t.content}" for t in self.history[-6:])\n            prompt = ANSWER_PROMPT.format(\n                sources=format_sources(reranked), history=history_text, question=standalone\n            )\n            answer = self.llm.invoke(prompt).content.strip()\n            sources = [\n                {"rank": i + 1, "source": c["source"], "page": c["page"], "score": round(s, 3), "text": c["text"]}\n                for i, (c, s) in enumerate(reranked)\n            ]\n\n        self.history.append(ChatTurn("user", question))\n        self.history.append(ChatTurn("assistant", answer))\n        return {"answer": answer, "sources": sources, "standalone_question": standalone}\n\n    def naive_chat(self, question):\n        """The Section 2 baseline, reusing the SAME index: top-1 dense-only, no rerank,\n        no guardrail, no citations, no memory. For side-by-side comparison only."""\n        order = self.index._dense_order(question)\n        top_chunk = self.index.chunks[order[0]]\n        prompt = f"Answer using this context:\\n\\n{top_chunk[\'text\']}\\n\\nQuestion: {question}\\nAnswer:"\n        answer = self.llm.invoke(prompt).content.strip()\n        return {"answer": answer, "source": {"source": top_chunk["source"], "page": top_chunk["page"]}}\n'

In [ ]:
out_dir = Path.cwd() / "production_rag_chatbot"
out_dir.mkdir(exist_ok=True)
(out_dir / "rag_pipeline.py").write_text(RAG_PIPELINE_SOURCE, encoding="utf-8")
print(f"Wrote {out_dir / 'rag_pipeline.py'} \u2014 ({len(RAG_PIPELINE_SOURCE.splitlines())} lines)")
print("Now run:  cd production_rag_chatbot && streamlit run app.py")

## 14. What's still missing — setting up the next class

Today's bot is genuinely production-*shaped*: hybrid retrieval, reranking, citations, a
groundedness guardrail, and memory. But three things are still missing before it's actually
production-*ready*:

- **We have no number for "how good."** Notebooks 09 (RAGAS) and 10 (DeepEval) — already in
  this folder — give you faithfulness, answer relevancy, and context precision/recall scores
  instead of "it felt fine when I tried it."
- **No observability, no resilience.** Notebook 11 covers persistent memory across restarts,
  streaming, fallback models, and tracing — what happens when the model API times out at 2am.
- **No regression protection.** Right now, changing `CHUNK_SIZE` or the rerank threshold and
  seeing if things got better or worse means testing by hand. `capstone_rag_studio/` in this
  repo shows the full-scale version: FastAPI backend, evaluation harness, guardrails, caching.

**In-class challenge — "Break my RAG":** now that your Streamlit app is running on your own
document, try to make it fail. Find a question it hallucinates on, or one where the guardrail
fires when it shouldn't. Share your best "gotcha" — it's the perfect motivation for why
Notebooks 09/10 exist.


## 15. Summary

- **Ingestion** dispatches by file extension so any uploaded PDF/DOCX/TXT/MD works.
- **Chunking** uses recursive splitting and keeps `source` + `page` on every chunk.
- **Retrieval** is hybrid: dense (Chroma + bge-small) and sparse (BM25), fused with RRF.
- **Reranking** applies a cross-encoder second pass over the fused candidates.
- **Generation** is grounded and cited — the model never answers off raw retrieval, only off
  the numbered, reranked sources.
- **Guardrail**: a low top rerank score means refuse, not hallucinate.
- **Memory**: a condense-question step rewrites follow-ups into standalone queries before retrieval.
- **Shipping**: the same class, unmodified, becomes `rag_pipeline.py` and powers a Streamlit
  chat app — `production_rag_chatbot/app.py`.
